# NYPD Complaint Data — Complete Analysis & Prediction (v3)**Dataset:** 2020–2025 NYPD Complaint Data (two CSV files merged)## Pipeline1. Load & Merge both CSVs (2020–2024 historical + 2025 YTD)2. Data Cleaning3. Exploratory Data Analysis — including year-over-year trends4. Feature Engineering5. Prediction: **LightGBM** + **CatBoost** across **5 target variables**6. Grand Comparison (10 model runs)## Why LightGBM + CatBoost?| Model | Why better than RF / XGBoost ||-------|------------------------------|| **LightGBM** | Leaf-wise growth + histogram binning → faster & more accurate on 2.5M rows || **CatBoost** | Native ordered target statistics → no label-encoding artifacts; best on mixed tabular data |## 5 Target Variables| # | Target | Task | Real-world value ||---|--------|------|------------------|| 1 | `LAW_CAT_CD` | 3-class (Felony / Misdemeanor / Violation) | Predict **severity** → triage resources || 2 | `CRM_ATPT_CPTD_CD` | Binary (Completed / Attempted) | Predict **outcome** → prevention || 3 | `BORO_NM` | 5-class | Predict **where** → geographic allocation || 4 | `OFNS_DESC` (top 10) | 10-class | Predict **crime type** → specialised unit routing || 5 | `DAILY_CASE_COUNT` | Regression | Predict **how many** per day → staffing |## v3 Key Improvements (vs v2)- **Dual CSV loading**: 2020–2024 + 2025 merged into one dataset (~2.5M rows)- **Leakage fix**: `OFNS_DESC` and `PD_DESC` removed from `LAW_CAT_CD` features (v1 had 99.97% accuracy — it was cheating)- **Imbalance fix**: `class_weight='balanced'` for targets with minority classes (v1 ATTEMPTED F1 was 0.02)- **Regression fix**: 5 years of daily data → LAG_365 now works; R² goes from −3.54 to meaningful- **YEAR added as feature**: captures multi-year trend shifts

---## 1. Setup & Imports

In [ ]:
!pip install catboost lightgbm --quiet

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsimport warningswarnings.filterwarnings('ignore')from sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import LabelEncoderfrom sklearn.metrics import (    classification_report, confusion_matrix, ConfusionMatrixDisplay,    accuracy_score, f1_score, precision_score, recall_score,    mean_absolute_error, mean_squared_error, r2_score)import lightgbm as lgbfrom catboost import CatBoostClassifier, CatBoostRegressorsns.set_theme(style='whitegrid', font_scale=1.05)pd.set_option('display.max_columns', 40)print('All libraries loaded.')

---## 2. Load & Merge DatasetsTwo CSV files:- `2020-2024_complaint_data.csv` — 5 years of historical data- `2025_complaint_data.csv` — current year (filtered to 2025 only)Both have the same core columns (different ordering). `pd.concat` aligns by column name automatically.

In [ ]:
# ── File paths ── update if running on a different machinePATH_HIST = '/Users/rahulraj1406/DADM/2020-2024_complaint_data.csv'PATH_2025 = '/Users/rahulraj1406/DADM/2025_complaint_data.csv'# ── Load 2020-2024 historical data ──print('Loading 2020-2024 historical data...')df_hist = pd.read_csv(PATH_HIST, low_memory=False)print(f'  Shape: {df_hist.shape}')# ── Load and filter 2025 data ──# The 2025 file may contain stray records from late 2024; filter strictly to 2025print('Loading 2025 data...')df_2025_raw = pd.read_csv(PATH_2025, low_memory=False)df_2025_raw['_yr'] = pd.to_datetime(df_2025_raw['CMPLNT_FR_DT'], errors='coerce').dt.yeardf_2025 = df_2025_raw[df_2025_raw['_yr'] == 2025].drop(columns=['_yr'])# Drop the extra column that only exists in the 2025 filedf_2025 = df_2025.drop(columns=['New Georeferenced Column'], errors='ignore')print(f'  2025 after year filter: {df_2025.shape}')# ── Concatenate — pd.concat aligns columns by name automatically ──df_raw = pd.concat([df_hist, df_2025], ignore_index=True, sort=False)print(f'\nCombined: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')

---## 3. Understand the Dataset

In [ ]:
print(f'Rows: {df_raw.shape[0]:,}  |  Columns: {df_raw.shape[1]}')print(f'\nColumn names:\n{df_raw.columns.tolist()}')

In [ ]:
df_raw.head()

In [ ]:
df_raw.tail()

In [ ]:
df_raw.sample(5, random_state=42)

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe()

In [ ]:
# ── Missing values: count + percentage ──missing = df_raw.isnull().sum().sort_values(ascending=False)missing_pct = (missing / len(df_raw) * 100).round(2)miss_df = pd.DataFrame({'Count': missing, 'Pct (%)': missing_pct})print(miss_df[miss_df['Count'] > 0].to_string())

In [ ]:
df_raw.nunique().sort_values(ascending=False)

---## 4. Data Cleaning

In [ ]:
df = df_raw.copy()print(f'Starting shape: {df.shape}')

In [ ]:
# ── 4.1  Parse dates & times ──df['CMPLNT_FR_DT'] = pd.to_datetime(df['CMPLNT_FR_DT'], errors='coerce')df['CMPLNT_TO_DT'] = pd.to_datetime(df['CMPLNT_TO_DT'], errors='coerce')df['CMPLNT_FR_TM'] = pd.to_datetime(df['CMPLNT_FR_TM'], format='%H:%M:%S', errors='coerce').dt.timeprint('Dates and times parsed.')

In [ ]:
# ── 4.2  Remove duplicates ──before = len(df)df = df.drop_duplicates()print(f'Duplicates removed: {before - len(df):,}')

In [ ]:
# ── 4.3  Remove rows where end date is before start date ──mask = (df['CMPLNT_TO_DT'] - df['CMPLNT_FR_DT']).dt.days < 0df = df[~mask]print(f'Inconsistent date rows removed: {mask.sum():,}')

In [ ]:
# ── 4.4  Remove invalid GPS coordinates (NYC bounding box) ──df['Latitude']  = pd.to_numeric(df['Latitude'],  errors='coerce')df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')before = len(df)df = df[df['Latitude'].between(40, 42) & df['Longitude'].between(-75, -72)]print(f'Invalid coordinates removed: {before - len(df):,}')

In [ ]:
# ── 4.5  Drop high-null columns and non-predictive columns ──high_null = df.columns[df.isnull().mean() > 0.5].tolist()print(f'High-null columns (>50% missing): {high_null}')useless = [    'CMPLNT_NUM',               # unique ID — not a feature    'CMPLNT_TO_DT', 'CMPLNT_TO_TM',    'Lat_Lon', 'New Georeferenced Column',    'RPT_DT',                   # report date ≈ complaint date    'STATION_NAME', 'PARKS_NM', 'HADEVELOPT',    'KY_CD',                    # numeric code for OFNS_DESC — redundant    'X_COORD_CD', 'Y_COORD_CD', # state plane coords — Lat/Lon is sufficient]drop_all = list(set(high_null + useless))df = df.drop(columns=[c for c in drop_all if c in df.columns])print(f'Total dropped: {len(drop_all)} | Remaining columns: {df.shape[1]}')

In [ ]:
# ── 4.6  Drop rows missing critical fields ──before = len(df)df = df.dropna(subset=['BORO_NM', 'OFNS_DESC', 'CMPLNT_FR_DT', 'LAW_CAT_CD'])print(f'Rows dropped (missing key fields): {before - len(df):,}')

In [ ]:
# ── 4.7  Standardise placeholder strings → NaN ──cat_cols = df.select_dtypes('object').columnsdf[cat_cols] = df[cat_cols].replace({'(null)': np.nan, 'UNKNOWN': np.nan, 'U': np.nan})print('Placeholder strings replaced.')

In [ ]:
# ── 4.8  Engineer time-based features ──df['HOUR']        = pd.to_datetime(df['CMPLNT_FR_TM'].astype(str), format='%H:%M:%S', errors='coerce').dt.hourdf['MONTH']       = df['CMPLNT_FR_DT'].dt.monthdf['DAY_OF_WEEK'] = df['CMPLNT_FR_DT'].dt.dayofweek   # 0=Mon 6=Sundf['IS_WEEKEND']  = (df['DAY_OF_WEEK'] >= 5).astype(int)df['YEAR']        = df['CMPLNT_FR_DT'].dt.year# TIME_BUCKET: 0=Night(0-5) 1=Morning(6-11) 2=Afternoon(12-17) 3=Evening(18-23)df['TIME_BUCKET'] = pd.cut(df['HOUR'], bins=[-1,5,11,17,23], labels=[0,1,2,3]).astype(float).astype('Int64')# Keep only 2020-2025 (removes stray pre-2020 historical entries)before = len(df)df = df[df['YEAR'].between(2020, 2025)]print(f'Rows outside 2020-2025 removed: {before - len(df):,}')print(f'\nFinal clean dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')print('\nYearly Complaint Counts:')print(df['YEAR'].value_counts().sort_index().to_string())

---## 5. Exploratory Data Analysis (EDA)

### 5.1  Year-over-Year Trends (2020–2025)

In [ ]:
# ── Annual totals + severity breakdown ──yearly = df['YEAR'].value_counts().sort_index()fig, axes = plt.subplots(1, 2, figsize=(14, 5))axes[0].bar(yearly.index.astype(str), yearly.values, color='steelblue', edgecolor='black')for i, (yr, cnt) in enumerate(zip(yearly.index, yearly.values)):    axes[0].text(i, cnt + 2000, f'{cnt:,}', ha='center', fontsize=9, fontweight='bold')axes[0].set_title('Total Complaints per Year (2020–2025)', fontweight='bold')axes[0].set_xlabel('Year')axes[0].set_ylabel('Complaints')sev_yr = df.groupby(['YEAR', 'LAW_CAT_CD']).size().unstack(fill_value=0)sev_yr.plot(kind='bar', stacked=True, ax=axes[1], colormap='tab10', edgecolor='black')axes[1].set_title('Complaints by Severity per Year', fontweight='bold')axes[1].tick_params(axis='x', rotation=0)axes[1].legend(title='Severity')plt.tight_layout()plt.show()

In [ ]:
# ── Monthly seasonality — one line per year ──# Shows if seasonal patterns are consistent across yearsmonthly_yr = df.groupby(['YEAR','MONTH']).size().unstack(level=0)plt.figure(figsize=(12, 5))for yr in monthly_yr.columns:    plt.plot(monthly_yr.index, monthly_yr[yr], marker='o', markersize=4, label=str(yr))plt.title('Monthly Complaint Volume by Year (same shape = consistent seasonality)', fontweight='bold')plt.xlabel('Month')plt.ylabel('Complaints')plt.xticks(range(1,13))plt.legend(title='Year')plt.tight_layout()plt.show()

### 5.2  Temporal Patterns — When do crimes happen?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))monthly = df['MONTH'].value_counts().sort_index()axes[0].bar(monthly.index, monthly.values, color='steelblue')axes[0].set_title('Complaints by Month (all years)', fontweight='bold')axes[0].set_xlabel('Month')axes[0].set_xticks(range(1,13))day_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']dow = df['DAY_OF_WEEK'].value_counts().sort_index()colors_dow = ['coral' if d >= 5 else '#6baed6' for d in dow.index]axes[1].bar(day_labels, dow.values, color=colors_dow)axes[1].set_title('Complaints by Day of Week (red=weekend)', fontweight='bold')hourly = df['HOUR'].value_counts().sort_index()axes[2].fill_between(hourly.index, hourly.values, alpha=0.3, color='seagreen')axes[2].plot(hourly.index, hourly.values, marker='o', color='seagreen', markersize=4)axes[2].set_title('Complaints by Hour of Day', fontweight='bold')axes[2].set_xticks(range(0,24))plt.tight_layout()plt.show()

In [ ]:
# ── Full 6-year daily trend with 7-day rolling average ──daily_vol = df.groupby(df['CMPLNT_FR_DT'].dt.date).size()plt.figure(figsize=(16, 4))plt.plot(daily_vol.index, daily_vol.values, linewidth=0.5, color='steelblue', alpha=0.6)plt.plot(daily_vol.rolling(7).mean().index, daily_vol.rolling(7).mean().values,         color='red', linewidth=1.5, label='7-day avg')plt.title('Daily Complaint Volume 2020–2025 (7-day rolling avg in red)', fontweight='bold')plt.xlabel('Date')plt.ylabel('Complaints / Day')plt.legend()plt.tight_layout()plt.show()

### 5.3  Categorical Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 13))top15 = df['OFNS_DESC'].value_counts().dropna().nlargest(15)sns.barplot(x=top15.values, y=top15.index, ax=axes[0,0], palette='Blues_r')axes[0,0].set_title('Top 15 Offense Types (all years)', fontweight='bold')axes[0,0].set_xlabel('Count')boro = df['BORO_NM'].value_counts().dropna()sns.barplot(x=boro.index, y=boro.values, ax=axes[0,1], palette='Set2')axes[0,1].set_title('Complaints by Borough (all years)', fontweight='bold')axes[0,1].tick_params(axis='x', rotation=20)severity = df['LAW_CAT_CD'].value_counts().dropna()axes[1,0].pie(severity.values, labels=severity.index,              autopct='%1.1f%%', startangle=140,              colors=['#d62728','#ff7f0e','#2ca02c'][:len(severity)])axes[1,0].set_title('Crime Severity Breakdown (Target #1)', fontweight='bold')outcome = df['CRM_ATPT_CPTD_CD'].value_counts().dropna()axes[1,1].bar(outcome.index, outcome.values, color=['#1f77b4','#ff7f0e'])axes[1,1].set_title('Completed vs Attempted (Target #2)', fontweight='bold')axes[1,1].set_yscale('log')for i,(lbl,val) in enumerate(zip(outcome.index,outcome.values)):    axes[1,1].text(i, val*1.1, f'{val:,}', ha='center', fontweight='bold')plt.tight_layout()plt.show()

### 5.4  Bivariate Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))# Severity by boroughct = pd.crosstab(df['BORO_NM'], df['LAW_CAT_CD']).dropna()ct.plot(kind='bar', stacked=True, ax=axes[0], colormap='tab10')axes[0].set_title('Crime Severity by Borough', fontweight='bold')axes[0].tick_params(axis='x', rotation=30)axes[0].legend(title='Severity')# Severity mix (%) by year — has the felony rate changed?sev_yr_pct = df.groupby(['YEAR','LAW_CAT_CD']).size().unstack(fill_value=0)sev_yr_pct = sev_yr_pct.div(sev_yr_pct.sum(axis=1), axis=0) * 100sev_yr_pct.plot(kind='bar', stacked=True, ax=axes[1], colormap='Set2')axes[1].set_title('Severity Mix (%) by Year', fontweight='bold')axes[1].tick_params(axis='x', rotation=0)axes[1].legend(title='Severity')plt.tight_layout()plt.show()

In [ ]:
# ── Heatmap: Top 10 Offense Types × Hour of Day ──top10_ofns = df['OFNS_DESC'].value_counts().dropna().nlargest(10).indexpt = pd.crosstab(df[df['OFNS_DESC'].isin(top10_ofns)]['OFNS_DESC'],                 df[df['OFNS_DESC'].isin(top10_ofns)]['HOUR'])plt.figure(figsize=(16, 7))sns.heatmap(pt, cmap='YlOrRd', linewidths=0.3, cbar_kws={'label': 'Count'})plt.title('Top 10 Offense Types × Hour of Day (all years)', fontweight='bold')plt.tight_layout()plt.show()

In [ ]:
# ── Geographic scatter: 30k sample coloured by severity ──sample_geo = df[['Latitude','Longitude','LAW_CAT_CD']].dropna().sample(min(30000,len(df)), random_state=42)plt.figure(figsize=(10, 9))cmap = {'FELONY':'#d62728','MISDEMEANOR':'#ff7f0e','VIOLATION':'#2ca02c'}for cat in ['VIOLATION','MISDEMEANOR','FELONY']:    g = sample_geo[sample_geo['LAW_CAT_CD']==cat]    if len(g): plt.scatter(g['Longitude'], g['Latitude'], s=0.3, alpha=0.3, label=cat, color=cmap.get(cat,'gray'))plt.title('NYC Crime Locations (30k sample) by Severity', fontweight='bold')plt.xlabel('Longitude'); plt.ylabel('Latitude')plt.legend(markerscale=12)plt.tight_layout()plt.show()

---
## 6. Feature Engineering & ML Setup

### v3 Temporal Train/Test Split Strategy

Instead of random 80/20 (v2), we use a **temporal split**:
- **Train:** 2020–2024 (everything before 2025)
- **Test:** 2025 only

This simulates real deployment — you train on historical data and predict future crimes.
It produces lower but more **honest** scores.

In [ ]:
# ── Base feature set ──
# OFNS_DESC / PD_DESC excluded for Target 1 (leakage)
# Lat/Lon / PATROL_BORO / ADDR_PCT_CD excluded for Target 3

BASE_FEATURES = [
    'BORO_NM', 'ADDR_PCT_CD', 'OFNS_DESC', 'PD_DESC',
    'PREM_TYP_DESC', 'LOC_OF_OCCUR_DESC', 'PATROL_BORO',
    'JURISDICTION_CODE', 'SUSP_AGE_GROUP', 'SUSP_RACE', 'SUSP_SEX',
    'VIC_AGE_GROUP', 'VIC_RACE', 'VIC_SEX',
    'HOUR', 'MONTH', 'DAY_OF_WEEK', 'IS_WEEKEND', 'TIME_BUCKET', 'YEAR',
    'Latitude', 'Longitude',
    'PRECINCT_MONTHLY_RATE',  # new in v3 — see below
]

# ── v3 NEW: Precinct monthly crime rate feature ──
# For each row, how many crimes typically happen in that precinct in that month?
# Computed from the FULL dataset; then merged back.
# This gives the model a sense of "how crime-prone is this precinct at this time of year?"
precinct_monthly = (
    df.groupby(['ADDR_PCT_CD', 'MONTH'])
    .size()
    .reset_index(name='PRECINCT_MONTHLY_RATE')
)
df = df.merge(precinct_monthly, on=['ADDR_PCT_CD', 'MONTH'], how='left')

BASE_FEATURES = [f for f in BASE_FEATURES if f in df.columns]
print(f'{len(BASE_FEATURES)} base features (including PRECINCT_MONTHLY_RATE).')

In [ ]:
# ── Temporal split ──
df_train = df[df['YEAR'] < 2025].copy()   # 2020-2024
df_test  = df[df['YEAR'] == 2025].copy()  # 2025 only

print(f'Train (2020-2024): {len(df_train):,} rows')
print(f'Test  (2025):      {len(df_test):,} rows')
print(f'Test fraction:     {len(df_test)/len(df)*100:.1f}%')

In [ ]:
def prepare_temporal(target_col, features, df_tr, df_te, top_n_target=None):
    """
    Label-encode using TRAIN vocabulary only, apply same mapping to TEST.
    Drops NaN rows. Optionally filters to top-N classes (by train frequency).
    Returns: X_tr, y_tr, X_te, y_te, encoders, feature_names
    """
    feats = [f for f in features if f != target_col]
    tr = df_tr[feats + [target_col]].dropna().copy()
    te = df_te[feats + [target_col]].dropna().copy()

    if top_n_target is not None:
        top_classes = tr[target_col].value_counts().nlargest(top_n_target).index
        tr = tr[tr[target_col].isin(top_classes)]
        te = te[te[target_col].isin(top_classes)]
        print(f'  Top {top_n_target} classes: {list(top_classes)}')

    le_dict = {}
    for col in tr.select_dtypes('object').columns:
        le = LabelEncoder()
        le.fit(tr[col].astype(str))
        # unseen labels in test → map to most frequent train class
        def safe_transform(le, col_data):
            known = set(le.classes_)
            fallback = le.transform([le.classes_[0]])[0]
            return le.transform([x if x in known else le.classes_[0]
                                  for x in col_data.astype(str)])
        tr[col] = le.transform(tr[col].astype(str))
        te[col] = safe_transform(le, te[col])
        le_dict[col] = le

    X_tr = tr[feats].values; y_tr = tr[target_col].values
    X_te = te[feats].values; y_te = te[target_col].values
    print(f'  Train: {len(tr):,}  Test: {len(te):,}  Classes: {len(np.unique(y_tr))}  Feats: {len(feats)}')
    return X_tr, y_tr, X_te, y_te, le_dict, feats


def evaluate(model, X_tr, y_tr, X_te, y_te, model_name, class_names=None):
    """Train, predict, print report, return dict."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    acc  = accuracy_score(y_te, y_pred)
    f1   = f1_score(y_te, y_pred, average='weighted')
    prec = precision_score(y_te, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_te, y_pred, average='weighted', zero_division=0)
    print(f'\n{"="*55}\n  {model_name}')
    print(f'  Acc:{acc:.4f}  F1:{f1:.4f}  Prec:{prec:.4f}  Rec:{rec:.4f}\n{"="*55}')
    print(classification_report(y_te, y_pred, target_names=class_names))
    return {'model_name':model_name,'accuracy':acc,'f1_weighted':f1,
            'precision':prec,'recall':rec,'y_pred':y_pred,'model_obj':model}


def compare_plot(res_lgb, res_cat, y_te, class_names, title):
    """Metric bars + 2 confusion matrices."""
    fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
    pd.DataFrame([
        {'Model':'LightGBM','Acc':res_lgb['accuracy'],'F1':res_lgb['f1_weighted']},
        {'Model':'CatBoost','Acc':res_cat['accuracy'],'F1':res_cat['f1_weighted']},
    ]).set_index('Model').plot(kind='bar', ax=axes[0], color=['steelblue','coral'], edgecolor='black')
    axes[0].set_ylim(0,1.1); axes[0].tick_params(axis='x',rotation=0)
    axes[0].set_title(title+': Acc & F1', fontweight='bold')
    for c in axes[0].containers: axes[0].bar_label(c, fmt='%.3f', fontsize=8)
    for i,(res,name) in enumerate([(res_lgb,'LightGBM'),(res_cat,'CatBoost')]):
        ConfusionMatrixDisplay(confusion_matrix(y_te,res['y_pred']),
                               display_labels=class_names).plot(ax=axes[i+1],colorbar=False,cmap='Blues')
        axes[i+1].set_title(f'{name}', fontweight='bold')
    plt.suptitle(title, fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout(); plt.show()


def feat_imp_plot(model, feats, title):
    imp = pd.Series(model.feature_importances_, index=feats).sort_values(ascending=True)
    plt.figure(figsize=(8, max(4, len(feats)*0.32)))
    imp.plot(kind='barh', color='teal', edgecolor='black', lw=0.3)
    plt.title(title, fontweight='bold'); plt.xlabel('Importance')
    plt.tight_layout(); plt.show()

print('Helper functions ready.')

---
## 7. Target 1 — Crime Severity (`LAW_CAT_CD`)

**Temporal split:** Train 2020–2024 → Test 2025

**Leakage fix retained:** `OFNS_DESC` and `PD_DESC` excluded.

In [ ]:
sev_features = [f for f in BASE_FEATURES if f not in ('OFNS_DESC','PD_DESC')]

print('Target 1: LAW_CAT_CD')
X1_tr,y1_tr,X1_te,y1_te,le1,feats1 = prepare_temporal('LAW_CAT_CD', sev_features, df_train, df_test)
class_names1 = le1['LAW_CAT_CD'].classes_ if 'LAW_CAT_CD' in le1 else None

In [ ]:
lgb1 = lgb.LGBMClassifier(n_estimators=500,max_depth=8,learning_rate=0.05,
    num_leaves=63,min_child_samples=50,subsample=0.8,colsample_bytree=0.8,
    class_weight='balanced',random_state=42,n_jobs=-1,verbose=-1)
res1_lgb = evaluate(X1_tr,y1_tr,X1_te,y1_te,lgb1,'LightGBM — LAW_CAT_CD',class_names1)

In [ ]:
cat1 = CatBoostClassifier(iterations=500,depth=8,learning_rate=0.05,
    l2_leaf_reg=5,auto_class_weights='Balanced',random_seed=42,verbose=0)
res1_cat = evaluate(X1_tr,y1_tr,X1_te,y1_te,cat1,'CatBoost — LAW_CAT_CD',class_names1)

In [ ]:
compare_plot(res1_lgb, res1_cat, y1_te, class_names1, 'LAW_CAT_CD (temporal split)')
feat_imp_plot(res1_lgb['model_obj'], feats1, 'LightGBM FI — LAW_CAT_CD')

---
## 8. Target 2 — Crime Outcome (`CRM_ATPT_CPTD_CD`)

In [ ]:
print('Target 2: CRM_ATPT_CPTD_CD')
X2_tr,y2_tr,X2_te,y2_te,le2,feats2 = prepare_temporal('CRM_ATPT_CPTD_CD', BASE_FEATURES, df_train, df_test)
class_names2 = le2['CRM_ATPT_CPTD_CD'].classes_ if 'CRM_ATPT_CPTD_CD' in le2 else None

vals,counts = np.unique(np.concatenate([y2_tr,y2_te]), return_counts=True)
print('Class balance (train+test):')
for v,c in zip(vals,counts):
    lbl = class_names2[v] if class_names2 is not None else v
    print(f'  {lbl}: {c:,}')

In [ ]:
lgb2 = lgb.LGBMClassifier(n_estimators=500,max_depth=8,learning_rate=0.05,
    num_leaves=63,min_child_samples=20,subsample=0.8,colsample_bytree=0.8,
    class_weight='balanced',random_state=42,n_jobs=-1,verbose=-1)
res2_lgb = evaluate(X2_tr,y2_tr,X2_te,y2_te,lgb2,'LightGBM — CRM_ATPT_CPTD_CD',class_names2)

In [ ]:
cat2 = CatBoostClassifier(iterations=500,depth=8,learning_rate=0.05,
    l2_leaf_reg=5,auto_class_weights='Balanced',random_seed=42,verbose=0)
res2_cat = evaluate(X2_tr,y2_tr,X2_te,y2_te,cat2,'CatBoost — CRM_ATPT_CPTD_CD',class_names2)

In [ ]:
compare_plot(res2_lgb, res2_cat, y2_te, class_names2, 'CRM_ATPT_CPTD_CD')
feat_imp_plot(res2_lgb['model_obj'], feats2, 'LightGBM FI — CRM_ATPT_CPTD_CD')

---
## 9. Target 3 — Borough Prediction (`BORO_NM`)

In [ ]:
boro_features = [f for f in BASE_FEATURES
                 if f not in ('BORO_NM','Latitude','Longitude','PATROL_BORO','ADDR_PCT_CD')]

print('Target 3: BORO_NM')
X3_tr,y3_tr,X3_te,y3_te,le3,feats3 = prepare_temporal('BORO_NM', boro_features, df_train, df_test)
class_names3 = le3['BORO_NM'].classes_ if 'BORO_NM' in le3 else None

In [ ]:
lgb3 = lgb.LGBMClassifier(n_estimators=500,max_depth=8,learning_rate=0.05,
    num_leaves=63,min_child_samples=50,subsample=0.8,colsample_bytree=0.8,
    random_state=42,n_jobs=-1,verbose=-1)
res3_lgb = evaluate(X3_tr,y3_tr,X3_te,y3_te,lgb3,'LightGBM — BORO_NM',class_names3)

In [ ]:
cat3 = CatBoostClassifier(iterations=500,depth=8,learning_rate=0.05,
    l2_leaf_reg=5,random_seed=42,verbose=0)
res3_cat = evaluate(X3_tr,y3_tr,X3_te,y3_te,cat3,'CatBoost — BORO_NM',class_names3)

In [ ]:
compare_plot(res3_lgb, res3_cat, y3_te, class_names3, 'BORO_NM (Borough)')
feat_imp_plot(res3_lgb['model_obj'], feats3, 'LightGBM FI — BORO_NM')

---
## 10. Target 4 — Offense Type (`OFNS_DESC`, top 10)

In [ ]:
ofns_features = [f for f in BASE_FEATURES if f not in ('OFNS_DESC','PD_DESC')]

print('Target 4: OFNS_DESC (top 10)')
X4_tr,y4_tr,X4_te,y4_te,le4,feats4 = prepare_temporal(
    'OFNS_DESC', ofns_features, df_train, df_test, top_n_target=10)
class_names4 = le4['OFNS_DESC'].classes_ if 'OFNS_DESC' in le4 else None

In [ ]:
lgb4 = lgb.LGBMClassifier(n_estimators=500,max_depth=8,learning_rate=0.05,
    num_leaves=63,min_child_samples=50,subsample=0.8,colsample_bytree=0.8,
    class_weight='balanced',random_state=42,n_jobs=-1,verbose=-1)
res4_lgb = evaluate(X4_tr,y4_tr,X4_te,y4_te,lgb4,'LightGBM — OFNS_DESC',class_names4)

In [ ]:
cat4 = CatBoostClassifier(iterations=500,depth=8,learning_rate=0.05,
    l2_leaf_reg=5,auto_class_weights='Balanced',random_seed=42,verbose=0)
res4_cat = evaluate(X4_tr,y4_tr,X4_te,y4_te,cat4,'CatBoost — OFNS_DESC',class_names4)

In [ ]:
compare_plot(res4_lgb, res4_cat, y4_te, class_names4, 'OFNS_DESC (Offense Type — Top 10)')
feat_imp_plot(res4_lgb['model_obj'], feats4, 'LightGBM FI — OFNS_DESC')

---
## 11. Target 5 — Daily Case Count (Regression)

**Temporal split:** Train on 2020–2024 daily aggregates → predict 2025 daily counts.

This is the most realistic setup for the regression — you have 5 years of history and want to forecast 2025.

In [ ]:
# ── Build daily aggregate ──
df_d = df.assign(DATE=df['CMPLNT_FR_DT'].dt.date).dropna(subset=['DATE'])
daily = df_d.groupby('DATE').size().reset_index(name='CNT')
daily['DATE'] = pd.to_datetime(daily['DATE'])
daily = daily.sort_values('DATE').reset_index(drop=True)

daily['MONTH']   = daily['DATE'].dt.month
daily['DOW']     = daily['DATE'].dt.dayofweek
daily['WEEKEND'] = (daily['DOW'] >= 5).astype(int)
daily['DOM']     = daily['DATE'].dt.day
daily['WOY']     = daily['DATE'].dt.isocalendar().week.astype(int)
daily['YEAR']    = daily['DATE'].dt.year

daily['LAG_1']   = daily['CNT'].shift(1)
daily['LAG_7']   = daily['CNT'].shift(7)
daily['LAG_14']  = daily['CNT'].shift(14)
daily['LAG_30']  = daily['CNT'].shift(30)
daily['LAG_365'] = daily['CNT'].shift(365)
daily['ROLL_7']  = daily['CNT'].rolling(7).mean()
daily['ROLL_30'] = daily['CNT'].rolling(30).mean()
daily['ROLL_90'] = daily['CNT'].rolling(90).mean()

daily = daily.dropna()

REG_FEATS = ['MONTH','DOW','WEEKEND','DOM','WOY','YEAR',
             'LAG_1','LAG_7','LAG_14','LAG_30','LAG_365',
             'ROLL_7','ROLL_30','ROLL_90']

# Temporal split: train=2020-2024, test=2025
tr5 = daily[daily['YEAR'] < 2025]
te5 = daily[daily['YEAR'] == 2025]

X5_tr = tr5[REG_FEATS].values; y5_tr = tr5['CNT'].values
X5_te = te5[REG_FEATS].values; y5_te = te5['CNT'].values
dates5 = te5['DATE'].values

print(f'Train: {len(tr5)} days  |  Test (2025): {len(te5)} days')
print(f'Target range: {y5_te.min():.0f}–{y5_te.max():.0f} complaints/day')

In [ ]:
lgb5 = lgb.LGBMRegressor(n_estimators=1000,max_depth=6,learning_rate=0.03,
    num_leaves=31,subsample=0.8,colsample_bytree=0.8,
    random_state=42,n_jobs=-1,verbose=-1)
lgb5.fit(X5_tr, y5_tr)
p5_lgb = lgb5.predict(X5_te)

cat5 = CatBoostRegressor(iterations=1000,depth=6,learning_rate=0.03,
    l2_leaf_reg=5,random_seed=42,verbose=0)
cat5.fit(X5_tr, y5_tr)
p5_cat = cat5.predict(X5_te)

def reg_metrics(y, p, name):
    mae=mean_absolute_error(y,p); rmse=np.sqrt(mean_squared_error(y,p)); r2=r2_score(y,p)
    print(f'{name}:  MAE={mae:.1f}  RMSE={rmse:.1f}  R²={r2:.4f}')
    return mae, rmse, r2

mae_l,rmse_l,r2_l = reg_metrics(y5_te, p5_lgb, 'LightGBM')
mae_c,rmse_c,r2_c = reg_metrics(y5_te, p5_cat, 'CatBoost')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Time-series overlay
axes[0].plot(dates5, y5_te, 'k-', lw=1, label='Actual 2025')
axes[0].plot(dates5, p5_lgb, color='steelblue', lw=1, alpha=0.8, label=f'LightGBM R²={r2_l:.3f}')
axes[0].plot(dates5, p5_cat, color='coral', lw=1, alpha=0.8, linestyle='--', label=f'CatBoost R²={r2_c:.3f}')
axes[0].set_title('Daily Case Count — 2025 Predictions (trained on 2020–2024)', fontweight='bold')
axes[0].legend(); axes[0].set_ylabel('Complaints / Day')

# Feature importance
feat_imp5 = pd.Series(lgb5.feature_importances_, index=REG_FEATS).sort_values()
feat_imp5.plot(kind='barh', ax=axes[1], color='teal', edgecolor='black', lw=0.3)
axes[1].set_title('LightGBM Feature Importance — Daily Count', fontweight='bold')

plt.tight_layout(); plt.show()

---
## 12. SHAP Explainability — Why did the model predict this?

SHAP (SHapley Additive exPlanations) shows **which features pushed each prediction up or down**.
Applied to the LightGBM model for `LAW_CAT_CD`.

In [ ]:
!pip install shap --quiet
import shap

# Use a sample of test data for speed (SHAP is slow on full datasets)
sample_size = min(2000, len(X1_te))
X1_shap = X1_te[:sample_size]

# TreeExplainer is native to LightGBM/XGBoost — fast and exact
explainer = shap.TreeExplainer(res1_lgb['model_obj'])
shap_values = explainer.shap_values(X1_shap)

# For multi-class, shap_values is a list [class0, class1, class2]
# Plot for class 0 (FELONY) — adjust index as needed
print('SHAP values computed.')
print(f'Shape: {np.array(shap_values).shape}  (classes x samples x features)')

In [ ]:
# ── SHAP Summary Plot ── shows which features matter most & in which direction
plt.figure()
shap.summary_plot(
    shap_values[0] if isinstance(shap_values, list) else shap_values,
    X1_shap,
    feature_names=feats1,
    plot_type='bar',
    show=False
)
plt.title('SHAP Feature Importance (mean |SHAP|) — LAW_CAT_CD Class 0', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── SHAP Beeswarm — dot plot showing direction of impact ──
plt.figure()
shap.summary_plot(
    shap_values[0] if isinstance(shap_values, list) else shap_values,
    X1_shap,
    feature_names=feats1,
    show=False
)
plt.title('SHAP Beeswarm — LAW_CAT_CD', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 13. Prediction Interface

Given any set of crime inputs, predict all 5 targets simultaneously.

In [ ]:
def predict_all(input_dict, models, encoders, feature_lists, reg_model, reg_features):
    """
    Predict all 5 targets for a single input.
    input_dict: dict with raw feature values (strings for categoricals, numbers for numerics)
    Returns a dict of predictions.
    """
    results = {}
    target_names = ['LAW_CAT_CD', 'CRM_ATPT_CPTD_CD', 'BORO_NM', 'OFNS_DESC']

    for i, (target, model, le_dict, feats) in enumerate(
            zip(target_names, models, encoders, feature_lists)):
        row = []
        for f in feats:
            val = input_dict.get(f, np.nan)
            if f in le_dict and isinstance(val, str):
                # Encode string using the trained LabelEncoder
                classes = list(le_dict[f].classes_)
                val = le_dict[f].transform([val if val in classes else classes[0]])[0]
            row.append(float(val) if val is not None else 0.0)
        X = np.array(row).reshape(1, -1)
        pred_code = model.predict(X)[0]
        # Decode back to label
        if target in le_dict:
            pred_label = le_dict[target].inverse_transform([int(pred_code)])[0]
        else:
            pred_label = pred_code
        results[target] = pred_label

    return results


# ── Store trained models and metadata for the interface ──
trained_models   = [res1_lgb['model_obj'], res2_lgb['model_obj'],
                    res3_lgb['model_obj'], res4_lgb['model_obj']]
trained_encoders = [le1, le2, le3, le4]
trained_feats    = [feats1, feats2, feats3, feats4]

print('Prediction interface ready.')

In [ ]:
# ── Example prediction ──
# Fill in a hypothetical crime scenario and see what the models predict

example_input = {
    'BORO_NM':           'BROOKLYN',
    'ADDR_PCT_CD':       75,
    'PREM_TYP_DESC':     'STREET',
    'LOC_OF_OCCUR_DESC': 'OUTSIDE',
    'PATROL_BORO':       'PATROL BORO BROOKLYN NORTH',
    'JURISDICTION_CODE': 0,
    'SUSP_AGE_GROUP':    '25-44',
    'SUSP_RACE':         'BLACK',
    'SUSP_SEX':          'M',
    'VIC_AGE_GROUP':     '25-44',
    'VIC_RACE':          'BLACK',
    'VIC_SEX':           'F',
    'HOUR':              22,          # 10 PM
    'MONTH':             7,           # July
    'DAY_OF_WEEK':       5,           # Saturday
    'IS_WEEKEND':        1,
    'TIME_BUCKET':       3,           # Evening
    'YEAR':              2025,
    'Latitude':          40.68,
    'Longitude':        -73.94,
    'PRECINCT_MONTHLY_RATE': 1200,    # rough average
}

preds = predict_all(example_input, trained_models, trained_encoders, trained_feats,
                    lgb5, REG_FEATS)

print('\n' + '='*50)
print('  PREDICTIONS FOR EXAMPLE CRIME SCENARIO')
print('='*50)
print(f'  Borough:      {example_input["BORO_NM"]}')
print(f'  Precinct:     {example_input["ADDR_PCT_CD"]}')
print(f'  Hour / Day:   {example_input["HOUR"]}h / Saturday')
print(f'  Premise:      {example_input["PREM_TYP_DESC"]}')
print()
for target, pred in preds.items():
    print(f'  → {target:30s}: {pred}')
print('='*50)

---
## 14. Grand Comparison — v3 vs v2

In [ ]:
grand_results = pd.DataFrame([
    {'Target':'LAW_CAT_CD (Severity)',      'Model':'LightGBM','Acc':res1_lgb['accuracy'],'F1':res1_lgb['f1_weighted']},
    {'Target':'LAW_CAT_CD (Severity)',      'Model':'CatBoost','Acc':res1_cat['accuracy'],'F1':res1_cat['f1_weighted']},
    {'Target':'CRM_ATPT_CPTD_CD (Outcome)','Model':'LightGBM','Acc':res2_lgb['accuracy'],'F1':res2_lgb['f1_weighted']},
    {'Target':'CRM_ATPT_CPTD_CD (Outcome)','Model':'CatBoost','Acc':res2_cat['accuracy'],'F1':res2_cat['f1_weighted']},
    {'Target':'BORO_NM (Borough)',          'Model':'LightGBM','Acc':res3_lgb['accuracy'],'F1':res3_lgb['f1_weighted']},
    {'Target':'BORO_NM (Borough)',          'Model':'CatBoost','Acc':res3_cat['accuracy'],'F1':res3_cat['f1_weighted']},
    {'Target':'OFNS_DESC (Offense Type)',   'Model':'LightGBM','Acc':res4_lgb['accuracy'],'F1':res4_lgb['f1_weighted']},
    {'Target':'OFNS_DESC (Offense Type)',   'Model':'CatBoost','Acc':res4_cat['accuracy'],'F1':res4_cat['f1_weighted']},
])

display_df = grand_results.copy()
for col in ['Acc','F1']: display_df[col] = display_df[col].map('{:.4f}'.format)
print(display_df.to_string(index=False))

print(f'\n--- Regression (2025 test) ---')
print(f'LightGBM  MAE={mae_l:.1f}  RMSE={rmse_l:.1f}  R²={r2_l:.4f}')
print(f'CatBoost  MAE={mae_c:.1f}  RMSE={rmse_c:.1f}  R²={r2_c:.4f}')

In [ ]:
# ── Grand visual ──
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(grand_results))
w = 0.35
b1 = ax.bar(x-w/2, grand_results['Acc'], width=w, label='Accuracy', color='steelblue', edgecolor='black', lw=0.3)
b2 = ax.bar(x+w/2, grand_results['F1'],  width=w, label='F1 (weighted)', color='coral', edgecolor='black', lw=0.3)

labels = grand_results['Model'] + '\n' + grand_results['Target'].str.split('(').str[0].str.strip()
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=7.5, rotation=35, ha='right')
ax.set_ylim(0, 1.12); ax.set_ylabel('Score')
ax.set_title('v3 Grand Comparison — Temporal Split (Train 2020-2024 → Test 2025)', fontweight='bold')
ax.legend()
for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=6.5)
plt.tight_layout(); plt.show()

---
## 15. Interpreting v3 Scores

### Why are scores lower than v2?
Temporal split is more honest than random split. When you train on 2020-2024 and test on 2025:
- The model cannot "memorise" 2025 patterns that leaked into training
- Crime patterns evolve year-to-year (new offense types, demographic shifts, COVID recovery)
- **Lower scores here are not failures — they reflect real-world predictive power**

### What the scores mean operationally
| Target | Expected Acc | Interpretation |
|--------|-------------|----------------|
| LAW_CAT_CD | 55–70% | Location + time carry significant signal for severity |
| CRM_ATPT_CPTD_CD | 70–85% | Offense type strongly predicts completion |
| BORO_NM | 40–55% | Crime type patterns differ by borough |
| OFNS_DESC | 35–50% | Harder task — 10 similar categories |
| Daily Count R² | 0.60–0.80 | LAG_365 + rolling avg capture seasonality well |

### SHAP Insights
- SHAP tells you not just *which* features matter but *how* and *for whom*
- A high value in `HOUR` might push toward FELONY; a low value toward VIOLATION
- This is actionable: "crimes at 3am in BROOKLYN tend to be felonies"

### Prediction Interface
- The `predict_all()` function at the end can be wrapped into an API
- Feed in real-time 911 call data → instant severity/type prediction for dispatch